# RSNA 2026 Knee MRI Abnormality Detection: slice features and weak labels

Part 2 of 2. Run Part 1 first for context. This notebook is self-contained (it rebuilds the gold/report-only split itself) and covers:

1. why a maximum-intensity projection (MIP) loses information (frozen-feature baseline, ~0.58 macro AUC);
2. per-slice frozen features with mean/max pooling across slices (~0.67);
3. weak labels from radiology reports for the 4,349 report-only studies, and blending them with the gold-only head (~0.71 at a fixed 0.5 weight).

All numbers are macro AUC over the 12 findings on the 58 gold studies (5-fold multilabel-stratified CV, pooled out-of-fold predictions). With 58 studies a single per-finding AUC carries roughly ±0.06 of noise, and design choices were made on the same studies, so treat the figures as optimistic.

## 0. Setup

In [ ]:
!pip install -q iterative-stratification pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg gdcm opencv-python-headless

In [ ]:
import os, re, time
import numpy as np
import pandas as pd
import cv2
import pydicom
from tqdm import tqdm

DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
# Output of src/extract_features.py saved as a Kaggle dataset (adjust to your dataset's path)
FEATS_PATH = "/kaggle/input/rsna-knee-slice-feats/all_slice_feats.npz"

train_df = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
train_series = pd.read_csv(os.path.join(DATA_ROOT, "train_series.csv"))

label_cols = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
              "Medial OA", "Lateral OA", "PF OA", "Effusion",
              "Synovitis", "Baker's", "Contusion", "Fracture"]
PLANES = ["Axial", "Coronal", "Sagittal"]

# 58 gold studies (all 12 labels present) and 4,349 report-only studies (no labels)
gold_df = train_df[train_df[label_cols].notna().all(axis=1)].reset_index(drop=True)
weak_df = train_df[train_df[label_cols].isna().all(axis=1)].reset_index(drop=True)
y = gold_df[label_cols].values.astype(np.float32)
print(f"Gold: {len(gold_df)}, report-only: {len(weak_df)}")

def normalize_image(img, lo_p=1, hi_p=99):
    lo, hi = np.percentile(img, lo_p), np.percentile(img, hi_p)
    if hi - lo < 1e-6:
        return np.zeros_like(img, dtype=np.float32)
    return ((np.clip(img, lo, hi) - lo) / (hi - lo)).astype(np.float32)

def load_resize_normalize(path, size=256):
    px = pydicom.dcmread(path).pixel_array.astype(np.float32)
    return normalize_image(cv2.resize(px, (size, size), interpolation=cv2.INTER_AREA))

def get_best_series_for_plane(study_rows, plane, preferred_fluid=1):
    """Prefer a fluid-sensitive series for the plane, otherwise any series of that plane."""
    rows = study_rows[study_rows["Anatomical_Plane"] == plane]
    if len(rows) == 0:
        return None
    pref = rows[rows["Fluid_Sensitive"] == preferred_fluid]
    if len(pref) > 0:
        return pref.iloc[0]["SeriesInstanceUID"]
    rest = rows[rows["Fluid_Sensitive"] != preferred_fluid]
    return rest.iloc[0]["SeriesInstanceUID"] if len(rest) > 0 else None

## 1. Baseline: frozen ResNet18 features on the MIP input
Freeze an ImageNet ResNet18, take its 512-d features for the cached `(3, 256, 256)` MIP of each gold study (`gold_X.npy`, built in Part 1 under "Cache the 58 gold-labeled studies"), and fit a regularized logistic regression per finding. Pooled out-of-fold macro AUC: **~0.58**, with most per-finding AUCs close to chance for small, localized findings (menisci, fracture, contusion).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

X = np.load("/kaggle/working/gold_X.npy")   # (58, 3, 256, 256)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Frozen ImageNet ResNet18 -> 512-d feature vector per study
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()

mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

@torch.no_grad()
def extract(arr, bs=16):
    feats = []
    for i in range(0, len(arr), bs):
        xb = torch.tensor(arr[i:i + bs], dtype=torch.float32)
        xb = ((xb - mean) / std).to(device)   # ImageNet normalization (the fine-tune run skipped this)
        feats.append(backbone(xb).cpu().numpy())
    return np.concatenate(feats)

F = extract(X)
print("Features:", F.shape)

# Same folds every time, so the C values are compared fairly
splits = list(MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(F, y))

def evaluate(C):
    fold_aucs = []
    oof = np.full(y.shape, np.nan)
    for tr, va in splits:
        sc = StandardScaler().fit(F[tr])
        Ftr, Fva = sc.transform(F[tr]), sc.transform(F[va])
        aucs = []
        for i in range(12):
            if len(np.unique(y[tr, i])) < 2:
                continue
            clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced").fit(Ftr, y[tr, i])
            oof[va, i] = clf.predict_proba(Fva)[:, 1]
            if len(np.unique(y[va, i])) > 1:
                aucs.append(roc_auc_score(y[va, i], oof[va, i]))
        fold_aucs.append(np.mean(aucs))
    # Pooled out-of-fold AUC per label: far steadier than ~12 studies per fold
    pooled = [roc_auc_score(y[:, i], oof[:, i]) if len(np.unique(y[:, i])) > 1 else np.nan
              for i in range(12)]
    return np.mean(fold_aucs), np.std(fold_aucs), np.array(pooled)

best = None
for C in [0.0005, 0.005, 0.05]:
    m, s, pooled = evaluate(C)
    print(f"C={C}: per-fold macro AUC {m:.3f} (+/- {s:.3f}) | pooled-OOF macro AUC {np.nanmean(pooled):.3f}")
    if best is None or np.nanmean(pooled) > np.nanmean(best[1]):
        best = (C, pooled)

print(f"\nPer-label pooled AUC at C={best[0]}:")
for col, a in zip(label_cols, best[1]):
    print(f"  {col:18s} {a:.3f}")


## 2. Per-slice features, pooled across slices
Instead of collapsing each series into one MIP, run the frozen ResNet18 on every central slice (middle 60%) of the selected series, then mean- and max-pool the slice features per plane. A finding visible on only a few slices survives a max-pool but is diluted in a MIP.

Result: **0.64 to 0.67** macro AUC (best: max-pool over 3 planes, `C=0.005`, ~0.669), with the largest gains on menisci and OA labels.

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

@torch.no_grad()
def series_feature(series_dir, central=(0.2, 0.8), bs=32):
    """Features of every central slice -> [mean over slices, max over slices] (1024-d)."""
    files = sorted(os.listdir(series_dir))
    lo, hi = int(len(files) * central[0]), max(int(len(files) * central[1]), int(len(files) * central[0]) + 1)
    slices = np.stack([load_resize_normalize(os.path.join(series_dir, f)) for f in files[lo:hi]])
    feats = []
    for i in range(0, len(slices), bs):
        xb = torch.tensor(slices[i:i + bs], dtype=torch.float32).unsqueeze(1).repeat(1, 3, 1, 1)
        feats.append(backbone(((xb - mean) / std).to(device)).cpu().numpy())
    f = np.concatenate(feats)
    return np.concatenate([f.mean(0), f.max(0)])

# (n_studies, 3 planes, 1024)
feats = np.zeros((len(gold_df), len(PLANES), 1024), dtype=np.float32)
for n, (_, row) in enumerate(tqdm(gold_df.iterrows(), total=len(gold_df))):
    sid = row["StudyInstanceUID"]
    study_rows = train_series[train_series["StudyInstanceUID"] == sid]
    for p, plane in enumerate(PLANES):
        ser = get_best_series_for_plane(study_rows, plane, preferred_fluid=1)
        if ser is not None:
            feats[n, p] = series_feature(os.path.join(DATA_ROOT, "train_series", sid, ser))
np.save("/kaggle/working/gold_slice_feats.npy", feats)

splits = list(MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(y, y))

def pooled_oof_auc(F, C):
    oof = np.full(y.shape, np.nan)
    for tr, va in splits:
        sc = StandardScaler().fit(F[tr])
        Ftr, Fva = sc.transform(F[tr]), sc.transform(F[va])
        for i in range(12):
            if len(np.unique(y[tr, i])) < 2:
                continue
            clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced").fit(Ftr, y[tr, i])
            oof[va, i] = clf.predict_proba(Fva)[:, 1]
    return np.array([roc_auc_score(y[:, i], oof[:, i]) if len(np.unique(y[:, i])) > 1 else np.nan
                     for i in range(12)])

# mean-pool = first 512 dims of each plane block, max-pool = last 512
sets = {
    "mean-pool, 3 planes": feats[:, :, :512].reshape(len(y), -1),
    "max-pool, 3 planes":  feats[:, :, 512:].reshape(len(y), -1),
    "mean+max, 3 planes":  feats.reshape(len(y), -1),
}
results = {}
for name, F in sets.items():
    for C in [0.0005, 0.005]:
        a = pooled_oof_auc(F, C)
        results[(name, C)] = a
        print(f"{name:22s} C={C}: pooled-OOF macro AUC {np.nanmean(a):.3f}")

best_key = max(results, key=lambda k: np.nanmean(results[k]))
print(f"\nBest: {best_key}. Per-label pooled AUC:")
for col, a in zip(label_cols, results[best_key]):
    print(f"  {col:18s} {a:.3f}")


## 3. Features for all 4,407 studies
Extracting the same features for every training study is the slow step (hours on Kaggle). Run `src/extract_features.py` once as its own notebook (**Save & Run All**), save the resulting `all_slice_feats.npz` as a Kaggle dataset, and attach it here. Set `FEATS_PATH` in the setup cell to its location.

In [ ]:
d = np.load(FEATS_PATH, allow_pickle=True)
print("study ids:", d["ids"].shape, "| features:", d["feats"].shape, "| planes missing/failed:", int((~d["ok"]).sum()))

## 4. Weak labels from radiology reports
Reports exist for the 4,349 unlabeled studies but not for the test set, so they can only help during training. Negation-aware keyword rules give each report a **three-state** label per finding: *stated* (1), *explicitly ruled out* (0), or *not mentioned* (excluded; silence is not absence).

For each finding a logistic regression is trained on the image features of the decided report-only studies and scored directly on the 58 gold studies (the two sets are disjoint, so no cross-validation is needed for this head). Its scores are then rank-blended with the gold-only head.

What the experiment showed:
- Weak-only heads matched or beat the gold-only head for effusion, MCL, medial OA, contusion and synovitis (synovitis had only ~20 explicit negatives, so treat it as fragile).
- They were roughly equal for Baker's, PF OA, ACL, fracture and medial meniscus.
- They were clearly worse for lateral meniscus and lateral OA (the lateral OA rule agreed with gold only ~36% of the time), so a blend is not free.
- A fixed 50/50 blend for all 12 labels gave a macro AUC of about **0.71** vs **0.669** gold-only. The best-weight-per-label figure the cell prints (~0.72) is optimistic because the weight is picked on the same 58 studies.

In [ ]:
import re
import time
import numpy as np
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

T0 = time.time()
def log(msg):
    print(f"[{time.time() - T0:6.1f}s] {msg}", flush=True)
log("cell started")
NEG = re.compile(r"\b(no|without|normal|intact|absent|negative|not|nor)\b", re.I)
NEGW = r"\b(no|without|negative|absent|not|nor|normal|unremarkable|free of|rule[sd]? out)\b"

def split_sents(text):
    return re.split(r"(?<=[.!?\n])\s+", text)

def label_term(text, term_re):
    """1 = stated, 0 = explicitly negated, None = silent. Used for findings that are simply present/absent."""
    seen_neg = False
    for sent in split_sents(text):
        if re.search(term_re, sent, re.I):
            if not NEG.search(sent):
                return 1
            seen_neg = True
    return 0 if seen_neg else None

def label_abn(text, term_re, abn_re, own=None, other=None):
    """Like label_term, but the finding is an abnormal word near a structure (e.g. 'tear' + 'meniscus').
    Negation only counts if the cue sits just BEFORE the abnormal word, so
    'tear of the posterior horn without extrusion' stays positive.
    own/other: skip sentences about the other side only (lateral vs medial)."""
    seen_normal = False
    for sent in split_sents(text):
        if not re.search(term_re, sent, re.I):
            continue
        if other and re.search(other, sent, re.I) and not re.search(own, sent, re.I):
            continue
        hits = list(re.finditer(abn_re, sent, re.I))
        pos = [m for m in hits if not re.search(NEGW, sent[max(0, m.start() - 40):m.start()], re.I)]
        if pos:
            return 1
        if hits or re.search(r"normal|intact|unremarkable|\bno\b", sent, re.I):
            seen_normal = True
    return 0 if seen_normal else None

MENISC_ABN = r"tear|rupture|lesion|radial|flap|bucket|degenerat|horizontal|vertical|maceration|extrusion|fray|signal"
COMP_ABN = r"chondromalacia|cartilage\s+(loss|thinning|defect|damage|irregularit|fissur|wear)|osteophyt|osteoarthr|degenerat|chondral|thinning|sclerosis"
SIDE = r"(compartment|femorotibial|tibiofemoral|femoral condyle|tibial plateau)"
RULES = {
    "Effusion":         ("term", r"effusion"),
    "Baker's":          ("term", r"baker|popliteal cyst"),
    "Fracture":         ("term", r"fracture"),
    "Contusion":        ("term", r"contusion|bone bruise|marrow edema"),
    "ACL":              ("abn", r"\bacl\b|anterior cruciate", r"tear|rupture|sprain|injur|partial|complete|strain", None, None),
    "MCL":              ("abn", r"\bmcl\b|medial collateral|tibial collateral", r"tear|rupture|sprain|injur|partial|complete|strain|grade|edema|thicken", None, None),
    "Medial Meniscus":  ("abn", r"menisc", MENISC_ABN, "medial", "lateral"),
    "Lateral Meniscus": ("abn", r"menisc", MENISC_ABN, "lateral", "medial"),
    "Medial OA":        ("abn", r"medial\b[^.]{0,40}" + SIDE, COMP_ABN, "medial", "lateral"),
    "Lateral OA":       ("abn", r"lateral\b[^.]{0,40}" + SIDE, COMP_ABN, "lateral", "medial"),
    "PF OA":            ("abn", r"patell|trochle", COMP_ABN, None, None),
    "Synovitis":        ("abn", r"synov", r"synovitis|thicken|hypertroph|prolifer|inflam|enhanc", None, None),
}

def apply_rule(text, rule):
    return label_term(text, rule[1]) if rule[0] == "term" else label_abn(text, *rule[1:])

# ---- features (max-pool, 3 planes: the best config on gold) ----
d = np.load(FEATS_PATH, allow_pickle=True)
id2row = {s: i for i, s in enumerate(d["ids"])}
F_all = d["feats"][:, :, 512:].reshape(len(d["ids"]), -1)
F_gold = F_all[[id2row[s] for s in gold_df["StudyInstanceUID"]]]
F_weak = F_all[[id2row[s] for s in weak_df["StudyInstanceUID"]]]
log(f"features loaded: gold {F_gold.shape}, weak {F_weak.shape}")

splits = list(MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(y, y))

def gold_oof(i, C=0.005):
    oof = np.full(len(y), np.nan)
    for tr, va in splits:
        if len(np.unique(y[tr, i])) < 2:
            continue
        sc = StandardScaler().fit(F_gold[tr])
        clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
        clf.fit(sc.transform(F_gold[tr]), y[tr, i])
        oof[va] = clf.predict_proba(sc.transform(F_gold[va]))[:, 1]
    return oof

def auc(i, s):
    return roc_auc_score(y[:, i], s) if len(np.unique(y[:, i])) > 1 else np.nan

rank = lambda v: rankdata(v) / len(v)
log("computing gold-only baseline")
gold_auc = np.array([auc(i, gold_oof(i)) for i in range(12)])
print(f"Gold-only baseline, 12-label macro AUC: {np.nanmean(gold_auc):.3f}\n")

sc_w = StandardScaler().fit(F_weak)
Xw, Xg = sc_w.transform(F_weak), sc_w.transform(F_gold)
WEIGHTS = [0.3, 0.5, 0.7]
blend_auc = gold_auc.copy()

print(f"{'label':17s} {'weak +/-':>11s} {'gold cov':>8s} {'gold agree':>10s} {'gold-only':>9s} {'weak-only':>9s} " +
      " ".join(f"blend{w}" for w in WEIGHTS))
for lab, rule in RULES.items():
    i = label_cols.index(lab)
    log(f"{lab}: applying rule to weak reports")
    wl = [apply_rule(t, rule) for t in weak_df["Report"]]
    dec = np.array([v is not None for v in wl])
    yw = np.array([v for v in wl if v is not None], dtype=int)
    gl = [apply_rule(t, rule) for t in gold_df["Report"]]
    gdec = np.array([v is not None for v in gl])
    cov = gdec.mean()
    agree = (np.array([v for v in gl if v is not None], dtype=int) == y[gdec, i]).mean() if gdec.any() else np.nan
    if yw.sum() < 20 or (1 - yw).sum() < 20:
        print(f"{lab:17s} too few weak positives/negatives ({int(yw.sum())}/{int((1 - yw).sum())}), skipped")
        continue
    clf = LogisticRegression(C=0.005, max_iter=2000, class_weight="balanced").fit(Xw[dec], yw)
    log(f"{lab}: head trained")
    s_weak = clf.decision_function(Xg)
    s_gold = gold_oof(i)
    a_w = auc(i, s_weak)
    a_b = [auc(i, w * rank(s_weak) + (1 - w) * rank(s_gold)) for w in WEIGHTS]
    blend_auc[i] = max(a_b)   # optimistic: best weight picked on the same 58 studies
    print(f"{lab:17s} {int(yw.sum()):5d}/{int((1 - yw).sum()):<5d} {cov:8.2f} {agree:10.2f} {gold_auc[i]:9.3f} {a_w:9.3f} " +
          " ".join(f"{a:8.3f}" for a in a_b))

print(f"\n12-label macro AUC: gold-only {np.nanmean(gold_auc):.3f} -> with best blend per rule label {np.nanmean(blend_auc):.3f} (optimistic)")


## 5. Takeaways and next steps
- MIPs discard localized findings; per-slice max-pooling recovered much of the signal.
- Reports are a noisy proxy for expert labels (about half of gold-positive effusions are never mentioned in their report) but still carry transferable signal for several findings.
- Evaluate with pooled out-of-fold AUC; per-fold AUC on ~12 studies is too noisy to compare models.
- Next: a stronger or domain-specific backbone, learned slice attention, all slices at higher resolution, and multilingual report extraction to label more of the report-only studies.

Competition data, reports and derived arrays are not included in this repository.